# 01. Análisis exploratorio de la calidad del aire

Este cuaderno revisa las mediciones horarias de PM2.5 de la estación Kennedy. Muestra su evolución, los promedios por hora y día, y dónde se concentraron los faltantes antes de imputar.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

carpeta = Path.cwd()
if not (carpeta / "data").exists():
    carpeta = carpeta.parent
datos = pd.read_csv(carpeta / "data" / "datos limpios.csv", parse_dates=["fecha_hora"])
datos["hora"] = datos["fecha_hora"].dt.hour
datos["mes"] = datos["fecha_hora"].dt.month
print(f"Filas horarias: {len(datos)}")
print(f"Periodo: {datos.fecha_hora.min()} a {datos.fecha_hora.max()}")
print(f"Faltantes antes de imputar: {datos.pm25_original.isna().sum()}")

## Serie de tiempo

Se grafica la medición original. Los espacios vacíos siguen visibles y no se confunden con valores interpolados.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(datos.fecha_hora, datos.pm25_original, linewidth=0.7, color="steelblue")
ax.set(title="PM2.5 horario — estación Kennedy", xlabel="Fecha", ylabel="PM2.5 (µg/m³)")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## Perfil diario promedio

El promedio usa mediciones originales disponibles. La tabla identifica las tres horas con mayor promedio, sin asumir de antemano a qué hora ocurre el pico.

In [ ]:
perfil_hora = datos.groupby("hora").pm25_original.mean()
print("Horas con mayor promedio observado:")
display(perfil_hora.sort_values(ascending=False).head(3).round(2))
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(perfil_hora.index, perfil_hora.values, marker="o", color="darkorange")
ax.set(title="Promedio de PM2.5 por hora", xlabel="Hora", ylabel="PM2.5 (µg/m³)", xticks=range(0, 24, 2))
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## Días laborales y fines de semana

Comparamos los promedios observados de lunes a viernes con los de sábado y domingo.

In [ ]:
datos["tipo_dia"] = datos.fecha_hora.dt.dayofweek.map(
    lambda dia: "Fin de semana" if dia >= 5 else "Día laboral"
)
promedio_tipo_dia = datos.groupby("tipo_dia").pm25_original.mean().reindex(["Día laboral", "Fin de semana"])
print(promedio_tipo_dia.round(2))
fig, ax = plt.subplots(figsize=(6, 4))
promedio_tipo_dia.plot(kind="bar", color=["slateblue", "seagreen"], ax=ax)
ax.set(title="PM2.5 promedio: laboral y fin de semana", xlabel="", ylabel="PM2.5 (µg/m³)")
ax.tick_params(axis="x", rotation=0)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## Registros sin medición

Estos conteos usan `pm25_original`, antes de imputar. Se muestran por mes y por hora del día.

In [ ]:
faltantes = datos[datos.pm25_original.isna()]
faltantes_mes = faltantes.groupby("mes").size().reindex(range(1, 8), fill_value=0)
faltantes_hora = faltantes.groupby("hora").size().reindex(range(24), fill_value=0)
meses = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul"]
fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
ejes[0].bar(meses, faltantes_mes.values, color="indianred")
ejes[0].set(title="Faltantes por mes", ylabel="Registros")
ejes[1].bar(faltantes_hora.index, faltantes_hora.values, color="darkseagreen")
ejes[1].set(title="Faltantes por hora", xlabel="Hora", ylabel="Registros", xticks=range(0, 24, 2))
for ax in ejes:
    ax.grid(axis="y", alpha=0.2)
print(f"Total de faltantes originales: {int(faltantes_mes.sum())}")
plt.tight_layout()
plt.show()